In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from PIL import Image
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.impute import SimpleImputer

In [2]:
# load & clean data
BASE_DIR = os.getcwd()
df = pd.read_csv(os.path.join(BASE_DIR, 'listings_clean.csv'))

# fix any remaining string columns
df['sqft'] = pd.to_numeric(df['sqft'].astype(str).str.replace(',', '').str.extract(r'(\d+\.?\d*)')[0], errors='coerce')
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df.dropna(subset=['price', 'img_path'])
df = df[df['img_path'].apply(os.path.exists)]

print(f"Listings with valid images: {len(df)}")
print(f"Price range: ${df['price'].min():,.0f} – ${df['price'].max():,.0f}")

Listings with valid images: 8406
Price range: $150,000 – $11,495,000


In [3]:
# dataset
class PropertyDataset(Dataset):
    def __init__(self, img_paths, prices, transform):
        self.img_paths = img_paths
        self.prices    = prices
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.img_paths[idx]).convert('RGB')
            img = self.transform(img)
        except:
            img = torch.zeros(3, 224, 224)
        price = torch.tensor(self.prices[idx], dtype=torch.float32)
        return img, price

In [4]:
# transforms
# add augmentation for training to help generalization
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [5]:
# prep data
df_valid = df[df['img_path'].notna() & df['img_path'].apply(os.path.exists)].copy()
df_valid['price'] = pd.to_numeric(df_valid['price'], errors='coerce')
df_valid = df_valid.dropna(subset=['price'])

# normalize prices to help training (predict in $100k units)
price_mean = df_valid['price'].mean()
price_std  = df_valid['price'].std()
df_valid['price_norm'] = (df_valid['price'] - price_mean) / price_std

# train/val split
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df_valid, test_size=0.2, random_state=42)

train_dataset = PropertyDataset(train_df['img_path'].tolist(), train_df['price_norm'].tolist(), train_transform)
val_dataset   = PropertyDataset(val_df['img_path'].tolist(),   val_df['price_norm'].tolist(),   val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

# build model
model = models.resnet50(weights='IMAGENET1K_V1')

# freeze all layer 
for param in model.parameters():
    param.requires_grad = False

# unfreeze the last ResNet block (layer4) + final FC
for param in model.layer4.parameters():
    param.requires_grad = True

# replace final classification layer with regression head
model.fc = nn.Sequential(
    nn.Linear(2048, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 1)   # single output = price
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
print(f"Using device: {device}")

Train: 6724 | Val: 1682
Using device: cpu


In [6]:
# training loop
# only optimize unfrozen parameters
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)
criterion = nn.MSELoss()

In [7]:
# training loop
EPOCHS = 10

for epoch in range(EPOCHS):
    # train
    model.train()
    train_loss = 0
    for imgs, prices in train_loader:
        imgs, prices = imgs.to(device), prices.to(device)
        optimizer.zero_grad()
        preds = model(imgs).squeeze(1)
        loss  = criterion(preds, prices)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # validate
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, prices in val_loader:
            imgs, prices = imgs.to(device), prices.to(device)
            preds = model(imgs).squeeze(1)
            val_loss += criterion(preds, prices).item()

    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)

    # convert loss back to dollars (denormalize)
    train_mae_dollars = (train_loss ** 0.5) * price_std
    val_mae_dollars   = (val_loss   ** 0.5) * price_std

    scheduler.step()
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | ~Val $RMSE: ${val_mae_dollars:,.0f}")

Epoch 01/10 | Train Loss: 0.8443 | Val Loss: 0.7695 | ~Val $RMSE: $1,167,289
Epoch 02/10 | Train Loss: 0.6025 | Val Loss: 0.7886 | ~Val $RMSE: $1,181,685
Epoch 03/10 | Train Loss: 0.3994 | Val Loss: 0.7376 | ~Val $RMSE: $1,142,888


KeyboardInterrupt: 

In [ ]:
# save fine-tuned model
torch.save(model.state_dict(), os.path.join(os.getcwd(), 'resnet50_finetuned.pth'))
print("Saved resnet50_finetuned.pth")